# 🛠️ Fase 3: Data Preparation
**Proyek:** Analisis Sentimen Ulasan Shopee — Bummi Tani  
**Tujuan:** Membersihkan data, melakukan preprocessing teks, pelabelan Lexicon, ekstraksi fitur TF-IDF, dan menyiapkan data latih/uji dengan SMOTE.

---

## 📦 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import re
import string
import os
import pickle
import joblib
import warnings
warnings.filterwarnings('ignore')

# NLP
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Feature Extraction
from sklearn.feature_extraction.text import TfidfVectorizer

# Imbalance & Splitting
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

print('✅ Semua library berhasil diimport.')

## 📂 2. Load Dataset Mentah

In [ ]:
# Path ke dataset mentah (satu level ke atas dari folder ini)
DATA_PATH = '../2-Data-Understanding/ulasan_bummitani.csv'
OUTPUT_DIR = '.'   # Simpan output di folder ini

df_raw = pd.read_csv(DATA_PATH)
print(f'✅ Dataset mentah dimuat: {df_raw.shape[0]} baris, {df_raw.shape[1]} kolom')
df_raw.head(3)

## 🧹 3. Pembersihan Data Awal

In [ ]:
df = df_raw.copy()

# Pilih hanya kolom yang relevan
df = df[['Ulasan Text']].copy()
df.columns = ['ulasan']

print(f'Jumlah data awal : {len(df)}')

# --- Hapus missing values ---
df.dropna(subset=['ulasan'], inplace=True)
print(f'Setelah hapus NaN: {len(df)}')

# --- Hapus ulasan yang hanya berisi angka/simbol/kosong ---
df = df[df['ulasan'].astype(str).str.strip().str.len() > 5]
print(f'Setelah hapus terlalu pendek: {len(df)}')

# --- Hapus duplikat ---
df.drop_duplicates(subset=['ulasan'], inplace=True)
print(f'Setelah hapus duplikat: {len(df)}')

# Reset index
df.reset_index(drop=True, inplace=True)
print(f'\n✅ Data bersih: {len(df)} baris tersisa.')

## ✍️ 4. Text Preprocessing

Langkah-langkah:
1. **Case Folding** — Ubah semua huruf menjadi huruf kecil
2. **Cleaning** — Hapus karakter non-alfabet, URL, angka, emoji
3. **Normalisasi** — Perbaiki kata-kata slang/tidak baku
4. **Tokenizing** — Pisah teks menjadi token kata
5. **Stopword Removal** — Hapus kata-kata tidak bermakna
6. **Stemming** — Ubah kata ke bentuk dasar (menggunakan Sastrawi)

In [ ]:
# ============================================================
# Kamus Normalisasi (kata tidak baku → kata baku)
# Tambahkan entri sesuai temuan di data Anda
# ============================================================
normalisasi_dict = {
    'baguss': 'bagus', 'bagussss': 'bagus', 'baguuuus': 'bagus',
    'enakk': 'enak', 'enakkkk': 'enak', 'enaakkk': 'enak',
    'murahh': 'murah', 'murahhh': 'murah',
    'cepett': 'cepat', 'cepett': 'cepat', 'gercep': 'cepat',
    'oke': 'baik', 'okee': 'baik', 'oks': 'baik',
    'mantap': 'bagus', 'mantul': 'bagus', 'mantapp': 'bagus', 'mantappp': 'bagus',
    'rekomended': 'direkomendasikan', 'recommended': 'direkomendasikan', 'recomen': 'direkomendasikan',
    'packing': 'kemasan', 'packingnya': 'kemasannya', 'packingan': 'kemasan',
    'seller': 'penjual', 'shopee': '',
    'tq': 'terima kasih', 'thx': 'terima kasih', 'thanks': 'terima kasih', 'makasih': 'terima kasih',
    'mkasih': 'terima kasih', 'tks': 'terima kasih',
    'gak': 'tidak', 'ga': 'tidak', 'ngga': 'tidak', 'nggak': 'tidak', 'g': 'tidak',
    'bgt': 'banget', 'bgtt': 'banget',
    'sy': 'saya', 'aku': 'saya',
    'udah': 'sudah', 'udh': 'sudah',
    'blm': 'belum', 'blom': 'belum',
    'krn': 'karena', 'karna': 'karena',
    'lg': 'lagi', 'lgi': 'lagi',
    'yg': 'yang',
    'dgn': 'dengan', 'dg': 'dengan',
    'spt': 'seperti',
    'sdh': 'sudah', 'sdg': 'sedang',
    'hrs': 'harus',
    'kl': 'kalau', 'klo': 'kalau', 'klu': 'kalau',
    'tp': 'tetapi', 'tpi': 'tetapi',
    'jg': 'juga', 'jga': 'juga',
    'dr': 'dari', 'dlm': 'dalam',
    'utk': 'untuk', 'tuk': 'untuk',
    'brg': 'barang', 'brng': 'barang',
    'pake': 'pakai', 'pk': 'pakai',
    'gimana': 'bagaimana', 'gmn': 'bagaimana',
    'produk': 'produk', 'item': 'barang',
    'lumer': 'lembut', 'lembutttt': 'lembut',
    'hitam': 'hitam', 'pekat': 'pekat',
    'tepung': 'tepung', 'ketan': 'ketan',
    'bolu': 'bolu', 'kue': 'kue',
    'wow': 'bagus', 'keren': 'bagus', 'top': 'bagus', 'top markotop': 'sangat bagus',
    'jelek': 'buruk', 'rusak': 'rusak', 'bocor': 'bocor', 'sobek': 'sobek',
    'kecewa': 'kecewa', 'buruk': 'buruk',
    'enak banget': 'sangat enak', 'enak bgt': 'sangat enak',
}

print(f'✅ Kamus normalisasi dimuat: {len(normalisasi_dict)} entri')

In [ ]:
# ============================================================
# Inisialisasi Sastrawi Stemmer
# ============================================================
factory = StemmerFactory()
stemmer = StemmerFactory().create_stemmer()

# Stopwords Indonesia
stop_words_id = set(stopwords.words('indonesian'))
# Tambahkan stopwords kustom yang tidak relevan untuk sentimen
custom_stopwords = {
    'seller', 'shopee', 'kurir', 'tokopedia', 'marketplace',
    'update', 'penilaian', 'baru', 'sampai', 'terima', 'kasih',
    'pesanan', 'paket', 'barang', 'beli', 'order', 'produk',
    'kak', 'kakak', 'min', 'kak', 'bang', 'bung', 'mas', 'mbak',
    'ya', 'yah', 'sih', 'deh', 'dong', 'nih', 'loh', 'lah',
    'aja', 'ajaa', 'jga', 'blm', 'belum', 'mau', 'buat', 'bikin',
    'udah', 'udh', 'sdh', 'sudah', 'jadi', 'tapi',
    'selalu', 'semua', 'lebih', 'paling', 'sangat', 'banget',
    'dari', 'untuk', 'dengan', 'yang', 'dan', 'di', 'ke', 'pada',
    'lagi', 'juga', 'tidak', 'ada', 'ini', 'itu',
    'alhamdulillah', 'insyaallah', 'semoga', 'aamiin',
    'next', 'repeat', 'kali', 'pertama', 'kedua',
}
stop_words_id.update(custom_stopwords)

print('✅ Stemmer Sastrawi dan stopwords siap.')

In [ ]:
# ============================================================
# Fungsi Preprocessing Lengkap
# ============================================================

def case_folding(text):
    """Ubah semua huruf menjadi lowercase."""
    return str(text).lower()

def cleaning(text):
    """Hapus URL, angka, karakter khusus, emoji, dan spasi berlebih."""
    # Hapus URL
    text = re.sub(r'http\S+|www\S+', '', text)
    # Hapus mention & hashtag
    text = re.sub(r'@\w+|#\w+', '', text)
    # Hapus emoji (karakter non-ASCII)
    text = text.encode('ascii', 'ignore').decode('ascii')
    # Hapus angka
    text = re.sub(r'\d+', '', text)
    # Hapus tanda baca
    text = re.sub(r'[^\w\s]', ' ', text)
    # Hapus underscore
    text = re.sub(r'_', ' ', text)
    # Hapus spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def normalisasi(text):
    """Normalisasi kata tidak baku menggunakan kamus normalisasi."""
    words = text.split()
    normalized = [normalisasi_dict.get(w, w) for w in words]
    return ' '.join(normalized).strip()

def tokenizing(text):
    """Tokenisasi teks menjadi list kata."""
    return word_tokenize(text)

def remove_stopwords(tokens):
    """Hapus stopwords dari list token."""
    return [t for t in tokens if t not in stop_words_id and len(t) > 1]

def stemming(tokens):
    """Stemming menggunakan Sastrawi."""
    return [stemmer.stem(t) for t in tokens]

def preprocessing_pipeline(text):
    """Pipeline preprocessing lengkap, mengembalikan string."""
    text = case_folding(text)
    text = cleaning(text)
    text = normalisasi(text)
    tokens = tokenizing(text)
    tokens = remove_stopwords(tokens)
    tokens = stemming(tokens)
    return ' '.join(tokens)

print('✅ Fungsi preprocessing didefinisikan.')

In [ ]:
# Uji fungsi pada 1 teks contoh
contoh = "Barang bagus banget! Pengirimannya cepett, packing aman. Seller amanah, next order lagi ya kak 👍"
print(f'Input  : {contoh}')
print(f'Output : {preprocessing_pipeline(contoh)}')

In [ ]:
# Terapkan preprocessing ke seluruh dataset (ini mungkin membutuhkan beberapa menit)
print('⏳ Menjalankan preprocessing... (mungkin memakan beberapa menit)')
df['ulasan_bersih'] = df['ulasan'].apply(preprocessing_pipeline)
print(f'✅ Preprocessing selesai! {len(df)} ulasan diproses.')

# Hapus hasil yang kosong setelah preprocessing
df = df[df['ulasan_bersih'].str.strip().str.len() > 0]
df.reset_index(drop=True, inplace=True)
print(f'Data setelah hapus teks kosong: {len(df)} baris')

# Tampilkan perbandingan
df[['ulasan', 'ulasan_bersih']].head(5)

## 🏷️ 5. Pelabelan menggunakan InSet Lexicon

Aturan pelabelan:
- Hitung total skor bobot kata positif dan negatif dari kamus InSet
- **Total > 0** → Label **Positif (1)**
- **Total ≤ 0** → Label **Negatif (0)**

InSet Lexicon tersedia di: https://github.com/masdevid/ID-Lexicon

In [ ]:
# ============================================================
# Load InSet Lexicon
# PASTIKAN file inset_lexicon_positive.tsv dan
# inset_lexicon_negative.tsv ada di folder ini,
# atau sesuaikan path-nya.
#
# Download dari: https://github.com/masdevid/ID-Lexicon
#   - positive_words.tsv : kata\tskor (misal: bagus\t3)
#   - negative_words.tsv : kata\tskor (misal: buruk\t-3)
# ============================================================

LEXICON_POS_PATH = 'inset_lexicon_positive.tsv'
LEXICON_NEG_PATH = 'inset_lexicon_negative.tsv'

lexicon_pos = {}
lexicon_neg = {}

if os.path.exists(LEXICON_POS_PATH):
    try:
        df_pos = pd.read_csv(LEXICON_POS_PATH, sep='\t', header=None, names=['kata', 'skor'])
        lexicon_pos = dict(zip(df_pos['kata'], df_pos['skor']))
        print(f'✅ Lexicon Positif dimuat: {len(lexicon_pos)} kata')
    except Exception as e:
        print(f'⚠️  Gagal load lexicon positif: {e}')
else:
    print(f'⚠️  File {LEXICON_POS_PATH} tidak ditemukan.')

if os.path.exists(LEXICON_NEG_PATH):
    try:
        df_neg = pd.read_csv(LEXICON_NEG_PATH, sep='\t', header=None, names=['kata', 'skor'])
        lexicon_neg = dict(zip(df_neg['kata'], df_neg['skor']))
        print(f'✅ Lexicon Negatif dimuat: {len(lexicon_neg)} kata')
    except Exception as e:
        print(f'⚠️  Gagal load lexicon negatif: {e}')
else:
    print(f'⚠️  File {LEXICON_NEG_PATH} tidak ditemukan.')

# Gabung kedua lexicon
lexicon_all = {**lexicon_neg, **lexicon_pos}
print(f'\n✅ Total kamus lexicon: {len(lexicon_all)} kata')

In [ ]:
# ============================================================
# FALLBACK: Kamus mini jika file InSet tidak tersedia
# (Hapus blok ini jika sudah punya file lexicon lengkap)
# ============================================================
if len(lexicon_all) == 0:
    print('⚠️  Menggunakan kamus lexicon mini sebagai fallback...')
    lexicon_all = {
        # POSITIF
        'bagus': 3, 'enak': 3, 'lembut': 2, 'halus': 2, 'wangi': 2,
        'cepat': 2, 'aman': 2, 'rapi': 2, 'murah': 2, 'terjangkau': 2,
        'memuaskan': 3, 'recommended': 3, 'mantap': 3, 'puas': 3,
        'baik': 2, 'senang': 2, 'suka': 2, 'rekomendasi': 3,
        'lezat': 3, 'nikmat': 3, 'empuk': 2, 'original': 2,
        'amanah': 2, 'sukses': 2, 'lancar': 2, 'sesuai': 1,
        'direkomendasikan': 3, 'pekat': 1, 'legit': 2, 'pulen': 2,
        'responsif': 2, 'premium': 2, 'fresh': 1,
        # NEGATIF
        'buruk': -3, 'jelek': -3, 'rusak': -3, 'bocor': -3, 'sobek': -3,
        'kecewa': -3, 'mengecewakan': -3, 'lama': -2, 'lambat': -2,
        'mahal': -2, 'berpasir': -1, 'kasar': -2, 'tidak': -1,
        'kurang': -1, 'susah': -1, 'sulit': -1, 'parah': -3,
        'apek': -2, 'tengik': -3, 'bau': -3, 'pelit': -2,
        'palsu': -3, 'bohong': -3, 'pucet': -2, 'hambar': -2,
    }
    print(f'✅ Kamus mini dimuat: {len(lexicon_all)} kata')

In [ ]:
# ============================================================
# Fungsi Pelabelan InSet Lexicon
# ============================================================

def hitung_skor_sentimen(text):
    """Hitung total skor sentimen berdasarkan InSet Lexicon."""
    tokens = str(text).split()
    total_skor = sum(lexicon_all.get(token, 0) for token in tokens)
    return total_skor

def labeling_sentimen(text):
    """Beri label berdasarkan skor: >0 → Positif(1), <=0 → Negatif(0)."""
    skor = hitung_skor_sentimen(text)
    return 1 if skor > 0 else 0

# Terapkan ke dataset
df['skor_sentimen'] = df['ulasan_bersih'].apply(hitung_skor_sentimen)
df['label'] = df['ulasan_bersih'].apply(labeling_sentimen)
df['sentimen'] = df['label'].map({1: 'Positif', 0: 'Negatif'})

print('✅ Pelabelan selesai.')
print('\n=== Distribusi Label ===')
label_dist = df['sentimen'].value_counts()
print(label_dist)
print(f'\nRatio Positif:Negatif = {label_dist.get("Positif",0)}:{label_dist.get("Negatif",0)}')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribusi label
colors = ['#27AE60', '#E74C3C']
label_dist.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white', rot=0)
axes[0].set_title('Distribusi Label Sentimen', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Sentimen')
axes[0].set_ylabel('Jumlah')
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                 str(int(bar.get_height())), ha='center', va='bottom', fontweight='bold')

# Distribusi skor
axes[1].hist(df['skor_sentimen'], bins=30, color='#3498DB', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--', linewidth=2, label='Batas (skor=0)')
axes[1].set_title('Distribusi Skor Sentimen Lexicon', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Skor Sentimen')
axes[1].set_ylabel('Frekuensi')
axes[1].legend()

plt.suptitle('Hasil Pelabelan InSet Lexicon', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Tampilkan contoh per kelas
print('=== Contoh Ulasan POSITIF ===')
for _, row in df[df['label'] == 1].head(3).iterrows():
    print(f'  Skor: {row["skor_sentimen"]:+d} | {row["ulasan_bersih"][:80]}...')

print('\n=== Contoh Ulasan NEGATIF ===')
for _, row in df[df['label'] == 0].head(3).iterrows():
    print(f'  Skor: {row["skor_sentimen"]:+d} | {row["ulasan_bersih"][:80]}...')

## 💾 6. Simpan Data Bersih ke CSV

In [ ]:
# Simpan dataset yang sudah dilabeli
ulasan_bersih_path = os.path.join(OUTPUT_DIR, 'ulasan_bersih.csv')
df.to_csv(ulasan_bersih_path, index=False)
print(f'✅ Dataset bersih disimpan: {ulasan_bersih_path}')
print(df[['ulasan', 'ulasan_bersih', 'skor_sentimen', 'sentimen']].head())

## 🔢 7. Feature Extraction — TF-IDF

In [ ]:
# Inisialisasi TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=3000,    # Batasi fitur agar tidak terlalu sparse
    ngram_range=(1, 2),   # Unigram + Bigram
    min_df=2,             # Kata harus muncul minimal di 2 dokumen
    sublinear_tf=True     # Gunakan log(tf) untuk mengurangi dominasi frekuensi tinggi
)

X = tfidf_vectorizer.fit_transform(df['ulasan_bersih'])
y = df['label'].values

print(f'✅ TF-IDF berhasil dibuat.')
print(f'   → Jumlah sampel  : {X.shape[0]}')
print(f'   → Jumlah fitur   : {X.shape[1]}')
print(f'\nTop 20 fitur TF-IDF:')
feature_names = tfidf_vectorizer.get_feature_names_out()
print(list(feature_names[:20]))

## ✂️ 8. Train-Test Split (80:20)

In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.2

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y   # Pertahankan proporsi kelas
)

print(f'✅ Data dibagi (80:20):')
print(f'   → X_train  : {X_train.shape}')
print(f'   → X_test   : {X_test.shape}')
print(f'   → y_train  : {y_train.shape} | Positif: {y_train.sum()}, Negatif: {(y_train==0).sum()}')
print(f'   → y_test   : {y_test.shape}  | Positif: {y_test.sum()}, Negatif: {(y_test==0).sum()}')

## ⚖️ 9. Oversampling dengan SMOTE

In [ ]:
# SMOTE hanya diterapkan pada DATA LATIH (bukan data uji!)
min_class_count = min(y_train.sum(), (y_train == 0).sum())
k_neighbors = min(5, min_class_count - 1)  # Hindari error jika kelas kecil

smote = SMOTE(
    random_state=RANDOM_STATE,
    k_neighbors=max(1, k_neighbors)
)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f'✅ SMOTE selesai diterapkan pada data latih:')
print(f'   → Sebelum SMOTE: {X_train.shape[0]} sampel')
print(f'   → Sesudah SMOTE: {X_train_smote.shape[0]} sampel')
print(f'   → Distribusi y_train_smote: Positif={y_train_smote.sum()}, Negatif={(y_train_smote==0).sum()}')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (data, title) in zip(axes, [
    (y_train, 'Sebelum SMOTE (Data Latih)'),
    (y_train_smote, 'Sesudah SMOTE (Data Latih)'),
]):
    unique, counts = np.unique(data, return_counts=True)
    labels = ['Negatif' if u == 0 else 'Positif' for u in unique]
    colors = ['#E74C3C', '#27AE60']
    bars = ax.bar(labels, counts, color=colors[:len(unique)], edgecolor='white')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Jumlah')
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h + 1, str(int(h)),
                ha='center', va='bottom', fontweight='bold')

plt.suptitle('Dampak SMOTE pada Distribusi Kelas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 💾 10. Simpan Artefak untuk Fase Modeling

In [ ]:
import scipy.sparse

artifacts = {
    'X_train_smote.pkl': X_train_smote,
    'y_train_smote.pkl': y_train_smote,
    'X_test.pkl': X_test,
    'y_test.pkl': y_test,
    'tfidf_vectorizer.pkl': tfidf_vectorizer,
}

for filename, obj in artifacts.items():
    path = os.path.join(OUTPUT_DIR, filename)
    joblib.dump(obj, path)
    print(f'  ✅ Disimpan: {path}')

print(f'\n🎉 Semua artefak berhasil disimpan di folder: {os.path.abspath(OUTPUT_DIR)}')
print('\n=== Ringkasan Artefak ===')
print(f'  X_train_smote : {X_train_smote.shape}')
print(f'  y_train_smote : {y_train_smote.shape}')
print(f'  X_test        : {X_test.shape}')
print(f'  y_test        : {y_test.shape}')
print(f'  tfidf_vectorizer : {X_train_smote.shape[1]} fitur')

## ✅ Ringkasan Fase Data Preparation

| Tahap | Detail |
|-------|--------|
| Data awal | 500 ulasan |
| Setelah cleaning | Hapus NaN, duplikat, teks terlalu pendek |
| Preprocessing | Case folding → Cleaning → Normalisasi → Tokenizing → Stopword Removal → Stemming (Sastrawi) |
| Pelabelan | InSet Lexicon: skor > 0 = Positif, ≤ 0 = Negatif |
| TF-IDF | max_features=3000, ngram_range=(1,2) |
| Split | 80% latih, 20% uji (stratified) |
| SMOTE | Diterapkan pada data latih untuk menyeimbangkan kelas |

> **Artefak tersimpan:** `X_train_smote.pkl`, `y_train_smote.pkl`, `X_test.pkl`, `y_test.pkl`, `tfidf_vectorizer.pkl`, `ulasan_bersih.csv`